In [1]:
import pandas as pd

orders = pd.read_csv("orders.csv")
order_products = pd.read_csv("order_products__prior.csv")
products = pd.read_csv("products.csv")

In [2]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [3]:
order_products.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [4]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [5]:
orders.columns

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order'],
      dtype='str')

In [6]:
order_products.columns

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')

In [7]:
products.columns

Index(['product_id', 'product_name', 'aisle_id', 'department_id'], dtype='str')

# Data Overview

The dataset contains order-level and product-level information. Product co-occurrence will be analyzed using order-product mappings, where each order represents a basket of items.

In [8]:
basket = order_products.groupby('order_id')['product_id'].apply(list)
basket.head()

order_id
2    [33120, 28985, 9327, 45918, 30035, 17794, 4014...
3    [33754, 24838, 17704, 21903, 17668, 46667, 174...
4    [46842, 26434, 39758, 27761, 10054, 21351, 225...
5    [13176, 15005, 47329, 27966, 23909, 48370, 132...
6                                [40462, 15873, 41897]
Name: product_id, dtype: object

# Basket Construction

Products were grouped by order_id to form baskets, where each order represents a collection of items purchased together.

In [9]:
top_products = order_products['product_id'].value_counts().head(50).index
top_products

Index([24852, 13176, 21137, 21903, 47209, 47766, 47626, 16797, 26209, 27845,
       27966, 22935, 24964, 45007, 39275, 49683, 28204,  5876,  8277, 40706,
        4920, 30391, 45066, 42265, 49235, 44632, 19057,  4605, 37646, 21616,
       17794, 27104, 30489, 31717, 27086, 44359, 28985, 46979,  8518, 41950,
       26604,  5077, 34126, 22035, 39877, 35951, 43352, 10749, 19660,  9076],
      dtype='int64', name='product_id')

In [10]:
filtered = order_products[order_products['product_id'].isin(top_products)]

In [12]:
basket_matrix = filtered.groupby(['order_id', 'product_id'])['product_id'] \
    .count().unstack().fillna(0)

basket_matrix = (basket_matrix > 0).astype(int)

In [13]:
basket_matrix.head()

product_id,4605,4920,5077,5876,8277,8518,9076,10749,13176,16797,...,44359,44632,45007,45066,46979,47209,47626,47766,49235,49683
order_id,,,,,,,,,,,,,,,,,,,,,
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
10,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
14,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


# Transaction Matrix

A binary transaction matrix was created using the most frequent products, where each row represents an order and each column indicates the presence of a product in that order.

In [14]:
%pip install mlxtend

     ---------------------------------------- 1.4/1.4 MB 5.0 MB/s eta 0:00:00
     ---------------------------------------- 36.6/36.6 MB 2.5 MB/s eta 0:00:00
     ---------------------------------------- 8.1/8.1 MB 6.6 MB/s eta 0:00:00
     -------------------------------------- 309.1/309.1 kB 6.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from mlxtend.frequent_patterns import apriori, association_rules

frequent_items = apriori(basket_matrix, min_support=0.01, use_colnames=True)
frequent_items.head()

c:\Users\Dell\Desktop\kachra\Rudder Assignments\venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.035165,frozenset({4605})
1,0.039741,frozenset({4920})
2,0.029229,frozenset({5077})
3,0.042171,frozenset({5876})
4,0.040861,frozenset({8277})


In [16]:
rules = association_rules(frequent_items, metric="lift", min_threshold=1)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({24852}),frozenset({4605}),0.227119,0.035165,0.010174,0.044794,1.273831,1.0,0.002187,1.010081,0.278137,0.040354,0.009980,0.167052
1,frozenset({4605}),frozenset({24852}),0.035165,0.227119,0.010174,0.289311,1.273831,1.0,0.002187,1.087510,0.222801,0.040354,0.080468,0.167052
2,frozenset({4920}),frozenset({24852}),0.039741,0.227119,0.011820,0.297428,1.309570,1.0,0.002794,1.100074,0.246174,0.046346,0.090970,0.174736
3,frozenset({24852}),frozenset({4920}),0.227119,0.039741,0.011820,0.052044,1.309570,1.0,0.002794,1.012978,0.305856,0.046346,0.012812,0.174736
4,frozenset({13176}),frozenset({5876}),0.182367,0.042171,0.010757,0.058988,1.398767,1.0,0.003067,1.017871,0.348670,0.050320,0.017557,0.157038


In [17]:
rules = rules.sort_values(by='lift', ascending=False)
top_rules = rules.head(5)
top_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
55,frozenset({22935}),frozenset({24964}),0.054513,0.052760,0.010608,0.194603,3.688436,1.0,0.007732,1.176115,0.770907,0.109744,0.149743,0.197836
54,frozenset({24964}),frozenset({22935}),0.052760,0.054513,0.010608,0.201069,3.688436,1.0,0.007732,1.183440,0.769480,0.109744,0.155006,0.197836
68,frozenset({26209}),frozenset({47626}),0.067586,0.073368,0.013170,0.194863,2.655960,1.0,0.008211,1.150900,0.668682,0.103065,0.131114,0.187185
69,frozenset({47626}),frozenset({26209}),0.073368,0.067586,0.013170,0.179507,2.655960,1.0,0.008211,1.136406,0.672854,0.103065,0.120033,0.187185
7,frozenset({5876}),frozenset({47209}),0.042171,0.102650,0.010211,0.242131,2.358794,1.0,0.005882,1.184043,0.601417,0.075856,0.155436,0.170802


In [18]:
product_dict = dict(zip(products['product_id'], products['product_name']))

def get_name(itemset):
    return [product_dict[i] for i in itemset]

top_rules['antecedents_names'] = top_rules['antecedents'].apply(get_name)
top_rules['consequents_names'] = top_rules['consequents'].apply(get_name)

top_rules[['antecedents_names', 'consequents_names']]

,antecedents_names,consequents_names
55,[Organic Yellow Onion],[Organic Garlic]
54,[Organic Garlic],[Organic Yellow Onion]
68,[Limes],[Large Lemon]
69,[Large Lemon],[Limes]
7,[Organic Lemon],[Organic Hass Avocado]


# Association Rules

- If a customer buys {Organic Yellow Onion}, then they are likely to buy {Organic Garlic}.
- If a customer buys {Organic Garlic}, then they are likely to buy {Organic Yellow Onion}.
- If a customer buys {Limes}, then they are likely to buy {Large Lemon}.
- If a customer buys {Large Lemon}, then they are likely to buy {Limes}.
- If a customer buys {Organic Lemon}, then they are likely to buy {Organic Hass Avocado}.

# Bundle Suggestions

Organic Yellow Onion + Organic Garlic can be bundled as a cooking essentials combo, as they are frequently used together in meal preparation.

Limes + Large Lemon can be offered as a citrus bundle, targeting customers purchasing fresh produce for beverages or recipes.